# Wavelet Denoising Walkthrough

| | |
|---|---|
| Question | What happens to a JEDI voltage-imaging trace between the raw recording and the trace that `curation.ipynb` loads |
| Code reviewed | `vnoiser/denoiser.py` (port), `ref/` (original code), `vnoiser/curation.py`, `vnoiser/dataset.py` |
| Data used | `data/stan112/stan112_expt10/PF/`, scan 46, domain `soma` (60 s at 1075 Hz) |
| What runs here | Every denoiser stage on that trace, each compared with the file the original run saved |
| Not in the repo | Pixel-to-dF/F extraction and the upstream peak detector. Behaviour marked **inferred** comes from saved parameters and saved outputs only |
| Runtime | About 2 min (wavelet transform about 50 s, components pickle about 10 s) |

Reading guide

- Tables: what the code does, in execution order, with the parameter values actually used
- Diagrams: data flow
- Code cells: the same steps run on real data; outputs are saved in this file

## 1. End-to-end map

```mermaid
flowchart TD
    A["Raw imaging: pixel time series per ROI"] --> B["Mean over pixels of one domain -> F"]
    B --> C["dF/F = (F - slow baseline) / slow baseline"]
    C --> D["z-score, sign flip"]
    D --> E["Stage 2: wavelet transform -> 100 frequency rows"]
    E --> F["Stage 3: group rows into 10 frequency bands"]
    F --> G["Stage 4: one trace per band + event windows per band"]
    G --> H["Stage 5: mask each band trace outside its windows"]
    H --> I["sum over bands = rescaled_signal"]
    I --> J[("denoised_trace_scans.pkl = real part of the sum")]
    D --> K["Stage 6: 1 Hz baseline of z"]
    K --> L[("denoised_trace_components.pkl: rescaled_signal, lp_FIR1Hz, ...")]
    I --> L
    J --> M["Upstream peak detector: 3-400 Hz band-pass, 3.5 SD (code not in repo)"]
    M --> N[("detected_events_peaks.pkl, detected_events_peaks_curated.pkl")]
    J --> O["curation.ipynb Load: one scan/domain"]
    O --> P["threshold + local maxima -> candidates"]
    P --> Q["snippets, seed template, cosine, PCA panel, Yes/No labels"]
    Q --> R[("PF/.curation/*_template_curation.json")]
```

| # | Stage | Code | Saved output in `PF/` | Read by `curation.ipynb` |
|---|---|---|---|---|
| 0 | pixels -> F -> dF/F | `ref/preprocessor.py`, `calc_dfof_gauss` (not in the package) | `test.h5  <scan>/dfof_raw` (8 domains x N) | no |
| 1 | z-score, sign flip | `ref/preprocessor.py` | `test.h5  <scan>/dfof_zscore` | no |
| 2 | wavelet transform | `Denoiser._cwt` | `cwts.h5  <scan>/cwt_dfof` (100 x N x 8, complex64) | no |
| 3 | group rows into bands | `FrequencyClusterer.run` | not saved | no |
| 4 | band traces + event windows | `WaveletReducer.reduce_and_threshold` | not saved | no |
| 5 | masks + sum | `AdaptiveThreshold.run` | `denoised_trace_components.pkl  [scan][domain]["rescaled_signal"]` | no |
| 6 | 1 Hz baseline | `Denoiser._fir_lowpass` | `denoised_trace_components.pkl  [...]["lp_FIR1Hz"]` | no |
| 7 | final stored trace | upstream: `real(rescaled_signal)`, baseline **not** added | `denoised_trace_scans.pkl  [scan][domain]` | **yes, the only trace input** |
| 8 | peak detector | not in repo (**inferred**) | `param_spike_detect.pkl`, `detected_events_peaks.pkl`, `..._curated.pkl` | no (`load_events=False`) |
| 9 | candidate detection + curation | `vnoiser/curation.py` | `PF/.curation/{manual,fast,slow}_template_curation.json` | written |

Other files in `PF/`

| File | Content | Role |
|---|---|---|
| `fs_scans.pkl` | `{scan: sampling rate in Hz}` | needed by the loader |
| `scanIDs_ROIs.pkl` | `scanID_spatial` (scan list), `domain_ROInumber` (domain -> ROI indices), `roi_list` | needed by the loader; defines domain names |
| `ds_behavior.pkl`, `lapTC_remap.pkl`, `PF_*.pkl`, `domains_*.pkl`, `NMF_*.pkl` | behaviour and place-field analysis | not part of denoising or curation |

## 2. Setup

- Loads one scan/domain: the stage-1 input (`test.h5`), the stored final trace (`denoised_trace_scans.pkl`), and the upstream intermediates when the large files are present
- Row order inside `test.h5` = `domain_ROInumber` order without `All_domains` and `bg`: row 0 = `soma1`, rows 1-7 = `apical1`-`apical7`
- Domain key differs between files: `soma1` in `test.h5`/components, `soma` in `denoised_trace_scans.pkl`

In [1]:
import collections
import pickle
import time
from pathlib import Path

import h5py
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from scipy.ndimage import gaussian_filter1d
from scipy.signal import butter, find_peaks, sosfiltfilt

from vnoiser import Denoiser
from vnoiser.denoiser import AdaptiveThreshold, thresConfig
from vnoiser.curation import (
    AUTO_TEMPLATE_THRESHOLD,
    candidate_pca_embedding,
    cosine_similarity_rows,
    lowpass_trace,
    threshold_event_calls,
    threshold_slider_scale,
)

pio.renderers.default = "plotly_mimetype"

DATA_ROOT = Path("../data")
EXPERIMENT = "stan112/stan112_expt10"
SCAN = "46"
DOMAIN = "soma"            # key inside denoised_trace_scans.pkl
DOMAIN_UPSTREAM = "soma1"  # key inside denoised_trace_components.pkl
DOMAIN_ROW = 0             # row inside test.h5 (domain_ROInumber order without All_domains and bg)
PF = DATA_ROOT / EXPERIMENT / "PF"
for required in ["denoised_trace_scans.pkl", "fs_scans.pkl", "test.h5"]:
    if not (PF / required).exists():
        raise FileNotFoundError(f"Expected {PF / required}")


def md_table(header, rows):
    lines = ["| " + " | ".join(str(h) for h in header) + " |",
             "|" + "|".join("---" for _ in header) + "|"]
    lines += ["| " + " | ".join(str(c) for c in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


class _NestedDict(collections.defaultdict):
    # Stand-in for the upstream nested_dict class referenced by the components pickle.
    def __init__(self, *args, **kwargs):
        super().__init__(_NestedDict)


class _Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if name == "nested_dict":
            return _NestedDict
        return super().find_class(module, name)


def load_pickle(path):
    with open(path, "rb") as handle:
        return _Unpickler(handle).load()


def downsample(x, y, max_points=4000):
    step = max(1, int(np.ceil(len(x) / max_points)))
    return x[::step], y[::step]


def corr(a, b):
    return float(np.corrcoef(a, b)[0, 1])


fs_hz = float(load_pickle(PF / "fs_scans.pkl")[SCAN])
stored_denoised = np.asarray(load_pickle(PF / "denoised_trace_scans.pkl")[SCAN][DOMAIN], dtype=float)
with h5py.File(PF / "test.h5", "r") as h5:
    dfof_raw = h5[f"{SCAN}/dfof_raw"][DOMAIN_ROW].astype(float)
    z = h5[f"{SCAN}/dfof_zscore"][DOMAIN_ROW].astype(float)
components_path = PF / "denoised_trace_components.pkl"
components = load_pickle(components_path)[SCAN][DOMAIN_UPSTREAM] if components_path.exists() else None
cwts_path = PF / "cwts.h5"

n_samples = z.size
t = np.arange(n_samples) / fs_hz
excerpt_center = int(np.argmax(stored_denoised))
half = int(2.5 * fs_hz)
EX = slice(max(0, excerpt_center - half), min(n_samples, excerpt_center + half))
t_ex = t[EX]

md_table(["Loaded object", "Source file", "Shape / value", "Meaning"], [
    ["fs_hz", "fs_scans.pkl", fs_hz, "samples per second"],
    ["dfof_raw", "test.h5", dfof_raw.shape, "stage 0 output"],
    ["z", "test.h5", z.shape, "stage 1 output = denoiser input"],
    ["stored_denoised", "denoised_trace_scans.pkl", stored_denoised.shape, "what curation.ipynb loads"],
    ["components", "denoised_trace_components.pkl", "-" if components is None else list(components.keys()), "upstream intermediates"],
    ["cwts.h5 present", "cwts.h5", cwts_path.exists(), "upstream wavelet coefficients"],
    ["plot excerpt", "-", f"{t_ex[0]:.1f} - {t_ex[-1]:.1f} s", "5 s around the largest stored peak"],
])

| Loaded object | Source file | Shape / value | Meaning |
|---|---|---|---|
| fs_hz | fs_scans.pkl | 1075.0 | samples per second |
| dfof_raw | test.h5 | (64517,) | stage 0 output |
| z | test.h5 | (64517,) | stage 1 output = denoiser input |
| stored_denoised | denoised_trace_scans.pkl | (64517,) | what curation.ipynb loads |
| components | denoised_trace_components.pkl | ['rescaled_signal', 'lp_FIR1Hz', 'lp_FIR100Hz', 'envelope_lp'] | upstream intermediates |
| cwts.h5 present | cwts.h5 | True | upstream wavelet coefficients |
| plot excerpt | - | 27.0 - 32.0 s | 5 s around the largest stored peak |

## 3. Stages 0-1: fluorescence -> dF/F -> z-score (upstream, `ref/preprocessor.py`)

| Step | Operation | Parameter | Output |
|---|---|---|---|
| 0a | ROIs of one domain concatenated, mean over pixels per frame | 3 ROIs per domain (`domain_ROInumber`) | F, one value per frame |
| 0b | slow baseline = Gaussian smooth of F | sigma = 1500 samples (1.4 s at 1075 Hz) | |
| 0c | dF/F = (F - baseline) / baseline | `calc_dfof_gauss` | `test.h5  dfof_raw` |
| 1a | first 1000 samples replaced by the mean of the rest | 1000 samples (0.93 s) | removes the filter start-up artefact |
| 1b | slower baseline = Gaussian smooth of dF/F | sigma = 5000 samples (4.7 s) | |
| 1c | z = -(dF/F - baseline) / SD(dF/F) | `negative=True` | `test.h5  dfof_zscore` |

- Sign flip: JEDI fluorescence falls when the membrane depolarises; after the flip, depolarisation is positive
- Stage 1a happens in place, so the saved `dfof_raw` already carries the replaced first second
- The check below rebuilds z from `dfof_raw` with these exact steps

In [2]:
m = dfof_raw.copy()
m[:1000] = np.mean(m[1000:])
baseline_stage1 = gaussian_filter1d(m, sigma=5000)
z_reproduced = -1.0 * (m - baseline_stage1) / np.std(m)

md_table(["Check", "Value"], [
    ["samples / duration", f"{n_samples} / {n_samples / fs_hz:.1f} s"],
    ["dfof_raw: first 1000 samples already constant", bool(np.allclose(dfof_raw[:1000], dfof_raw[0]))],
    ["dfof_raw SD", f"{dfof_raw.std():.4f}"],
    ["z rebuilt from dfof_raw, max abs difference to stored z", f"{np.abs(z_reproduced - z).max():.1e}"],
    ["z mean / SD", f"{z.mean():.3f} / {z.std():.3f}"],
    ["z min / max", f"{z.min():.2f} / {z.max():.2f}"],
    ["correlation(z, dfof_raw) (negative = sign flipped)", f"{corr(z, dfof_raw):.3f}"],
])

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=("Stage 0 output: dF/F (dfof_raw) and the stage-1 baseline",
                                    "Stage 1 output: z (dfof_zscore)"))
x, y = downsample(t, dfof_raw)
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name="dF/F", line={"width": 1, "color": "#4c78a8"}), row=1, col=1)
x, y = downsample(t, baseline_stage1)
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name="baseline (sigma 5000 samples)", line={"width": 2, "color": "#e45756"}), row=1, col=1)
x, y = downsample(t, z)
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name="z", line={"width": 1, "color": "#333333"}), row=2, col=1)
fig.add_vrect(x0=t_ex[0], x1=t_ex[-1], fillcolor="orange", opacity=0.15, line_width=0, row="all", col=1)
fig.update_xaxes(title_text="time (s)", row=2, col=1)
fig.update_yaxes(title_text="dF/F", row=1, col=1)
fig.update_yaxes(title_text="z", row=2, col=1)
fig.update_layout(height=520, template="plotly_white", title="Stages 0-1, full recording (shaded = excerpt used below)")
fig.show()

| Check | Value |
|---|---|
| samples / duration | 64517 / 60.0 s |
| dfof_raw: first 1000 samples already constant | True |
| dfof_raw SD | 0.0042 |
| z rebuilt from dfof_raw, max abs difference to stored z | 4.4e-16 |
| z mean / SD | 0.000 / 1.000 |
| z min / max | -7.16 / 3.68 |
| correlation(z, dfof_raw) (negative = sign flipped) | -1.000 |

## 4. Stage 2: wavelet transform (`Denoiser._cwt`)

Plain description

- One wavelet row = the trace passed through one band-pass filter centred on one frequency. 100 rows cover 1075 Hz down to 1.08 Hz, log-spaced
- Filter width scales with frequency (measured for this wavelet): time width 0.35 / f seconds, frequency width 0.23 x f. Fast rows are sharp in time and wide in frequency; slow rows the reverse
- Each value is complex. Magnitude = how strong that frequency is at that time. Real part = the band-filtered signal itself
- Stage 3 groups rows using the magnitude. Stages 4-5 rebuild the trace using the real part

| Parameter | Value | Meaning |
|---|---|---|
| function | `pywt.cwt` | |
| wavelet | `cmor0.5-1.0` | complex Morlet; bandwidth 0.5, centre frequency 1.0 |
| scales | 100 values, log-spaced 1 ... 1000 | row frequency = fs / scale |
| sampling_period | 1 / fs | affects only the frequency labels, not the coefficients |
| output | `coeff` (100 x N, complex), `freqs` (100) | rows ordered high -> low frequency |
| saved upstream | `cwts.h5  <scan>/cwt_dfof` (100 x N x 8 domains, complex64), `cwt_freq`, `freq_scale` | |

In [3]:
model = Denoiser(fs=fs_hz)  # defaults = the settings the upstream files were made with (checked below)
t0 = time.time()
coeff, freqs = model._cwt(z)
cwt_seconds = time.time() - t0
scales = model.freq_scales

md_table(["row", "scale", "centre frequency (Hz)", "time width, 1 SD (ms)", "frequency width, 1 SD (Hz)"], [
    [r, f"{scales[r]:.1f}", f"{freqs[r]:.2f}", f"{1000 * 0.354 / freqs[r]:.2f}", f"{0.225 * freqs[r]:.2f}"]
    for r in [0, 25, 50, 75, 99]
])

checks = [
    ["coeff shape / dtype", f"{coeff.shape} / {coeff.dtype}"],
    ["compute time (s)", f"{cwt_seconds:.1f}"],
    ["frequency range (Hz)", f"{freqs.min():.2f} - {freqs.max():.1f}"],
]
if cwts_path.exists():
    with h5py.File(cwts_path, "r") as h5:
        stored_coeff = h5[f"{SCAN}/cwt_dfof"][:, :, DOMAIN_ROW]
        stored_freqs = h5[f"{SCAN}/cwt_freq"][:, DOMAIN_ROW]
        stored_scales = h5[f"{SCAN}/freq_scale"][:]
    checks += [
        ["stored cwts.h5 slice shape / dtype", f"{stored_coeff.shape} / {stored_coeff.dtype}"],
        ["stored scales equal to the defaults", bool(np.allclose(stored_scales, scales))],
        ["max abs difference, fresh vs stored coefficients", f"{np.abs(coeff - stored_coeff).max():.1e} (float32 rounding)"],
        ["max abs difference, frequency labels (Hz)", f"{np.abs(freqs - stored_freqs).max():.2f} (labels only; consistent with a sampling period rounded to 0.93 ms upstream)"],
    ]
    del stored_coeff
md_table(["Check", "Value"], checks)

power = np.abs(coeff[:, EX])
col_step = max(1, int(np.ceil(power.shape[1] / 700)))
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.3, 0.7], vertical_spacing=0.08,
                    subplot_titles=("z (excerpt)", "|coefficient| per row; y = row centre frequency"))
fig.add_trace(go.Scatter(x=t_ex, y=z[EX], mode="lines", name="z", line={"width": 1, "color": "#333333"}), row=1, col=1)
fig.add_trace(go.Heatmap(x=t_ex[::col_step], y=freqs, z=power[:, ::col_step], colorscale="Viridis",
                         colorbar={"title": "|coeff|", "len": 0.6, "y": 0.32}), row=2, col=1)
fig.update_yaxes(type="log", title_text="frequency (Hz)", row=2, col=1)
fig.update_yaxes(title_text="z", row=1, col=1)
fig.update_xaxes(title_text="time (s)", row=2, col=1)
fig.update_layout(height=620, template="plotly_white", title="Stage 2: wavelet magnitude of the excerpt")
fig.show()

| row | scale | centre frequency (Hz) | time width, 1 SD (ms) | frequency width, 1 SD (Hz) |
|---|---|---|---|---|
| 0 | 1.0 | 1075.00 | 0.33 | 241.88 |
| 25 | 5.7 | 187.86 | 1.88 | 42.27 |
| 50 | 32.7 | 32.83 | 10.78 | 7.39 |
| 75 | 187.4 | 5.74 | 61.71 | 1.29 |
| 99 | 1000.0 | 1.07 | 329.30 | 0.24 |

| Check | Value |
|---|---|
| coeff shape / dtype | (100, 64517) / complex128 |
| compute time (s) | 41.1 |
| frequency range (Hz) | 1.07 - 1075.0 |
| stored cwts.h5 slice shape / dtype | (100, 64517) / complex64 |
| stored scales equal to the defaults | True |
| max abs difference, fresh vs stored coefficients | 2.6e-07 (float32 rounding) |
| max abs difference, frequency labels (Hz) | 0.27 (labels only; consistent with a sampling period rounded to 0.93 ms upstream) |

## 5. Stage 3: group the 100 rows into frequency bands (`FrequencyClusterer.run`)

| Step | Operation | Parameter (`ClusteringConfig`) |
|---|---|---|
| 3a | standardise each row over time: subtract its mean, divide by its SD (complex arithmetic) | `standardize=True` |
| 3b | take the magnitude | |
| 3c | PCA across the 100 rows; each row is one observation with N time samples as features | `n_components=30` |
| 3d | hierarchical clustering, Ward linkage, on the first 10 PCA scores of each row | `n_comp_clu=10`, `method_linkage="ward"` |
| 3e | cut the tree into 5 clusters (`labels`) and 10 sub-clusters (`sublabels`) | `n_clusters=5`, `n_subclusters=10`, `method_fclust="maxclust"` |
| out | `sublabels`, `subfreqs` = 10 bands | `cwtReducerConfig.cluster_label="sublabels"` selects the 10-band cut downstream |

- Rows whose magnitude rises and falls together over time end up in the same band. Neighbouring rows overlap heavily, so bands come out as contiguous frequency ranges
- The 5-cluster cut is computed but not used
- Band ids are tree labels, not frequency order

In [4]:
cluster = model._cluster(coeff, freqs)
sublabels = cluster["sublabels"]
explained = cluster["explained"]

md_table(["band id (sublabel)", "rows", "frequency range (Hz)", "rows adjacent"], [
    [int(band_id), int(np.sum(sublabels == band_id)),
     f"{freqs[sublabels == band_id].min():.1f} - {freqs[sublabels == band_id].max():.1f}",
     "yes" if np.all(np.diff(np.flatnonzero(sublabels == band_id)) == 1) else "no"]
    for band_id in np.unique(sublabels)
])
md_table(["Check", "Value"], [
    ["PCA components kept / used for clustering", f"{len(explained)} / {model.cfg_clust.n_comp_clu}"],
    ["variance explained by the 10 components used", f"{explained[:10].sum():.3f}"],
    ["5-cluster cut sizes (unused)", np.bincount(cluster["labels"])[1:].tolist()],
    ["10-band cut sizes (used)", np.bincount(sublabels)[1:].tolist()],
])

fig = go.Figure(go.Scatter(x=freqs, y=sublabels, mode="markers",
                           marker={"color": sublabels, "colorscale": "Turbo", "size": 8},
                           text=[f"row {i}" for i in range(len(freqs))], name="rows"))
fig.update_xaxes(type="log", title_text="row centre frequency (Hz)")
fig.update_yaxes(title_text="band id (sublabel)", dtick=1)
fig.update_layout(height=380, template="plotly_white", title="Stage 3: band membership of each row")
fig.show()

| band id (sublabel) | rows | frequency range (Hz) | rows adjacent |
|---|---|---|---|
| 1 | 8 | 6.6 - 10.8 | yes |
| 2 | 11 | 3.1 - 6.2 | yes |
| 3 | 15 | 1.1 - 2.9 | yes |
| 4 | 9 | 11.5 - 20.1 | yes |
| 5 | 5 | 266.3 - 352.0 | yes |
| 6 | 16 | 377.5 - 1075.0 | yes |
| 7 | 6 | 21.6 - 30.6 | yes |
| 8 | 9 | 32.8 - 57.4 | yes |
| 9 | 9 | 61.5 - 107.5 | yes |
| 10 | 12 | 115.3 - 248.3 | yes |

| Check | Value |
|---|---|
| PCA components kept / used for clustering | 30 / 10 |
| variance explained by the 10 components used | 0.807 |
| 5-cluster cut sizes (unused) | [19, 24, 21, 15, 21] |
| 10-band cut sizes (used) | [8, 11, 15, 9, 5, 16, 6, 9, 9, 12] |

## 6. Stage 4: one trace per band and its event windows (`WaveletReducer.reduce_and_threshold`)

Per band, in band-id order

| Step | Operation | Rule (`cwtReducerConfig`) |
|---|---|---|
| 4a | band trace = mean of the band's rows, real part | |
| 4b | smoothing = running straight-line fit (Savitzky-Golay, order 1) | window 41 samples if max f <= 15 Hz; 21 if <= 50 Hz; 11 if <= 75 Hz; none above 75 Hz |
| 4c | rescale the smoothed trace so its maximum equals the pre-smoothing maximum | port only; `ref/` did not |
| 4d | open / close levels from the SD of the band trace | max f < 30 Hz: open 2.0 SD, close 1.0 SD (`slow_upthres`); otherwise open 2.5 SD, close 0 (`fast_upthres`) |
| 4e | `trigger_detect`: a window opens when the trace rises through the open level and closes when it falls through the close level | open/close must alternate; a window still open at the end closes at the last sample |
| 4f | drop the band if it has no windows | |
| out | `reduced_coeff` (bands x N), `reduced_freqs` (min, max Hz), `reduced_scale` (mean scale), `event_onsets[name] = {crossings (n x 2), iei, num_event}` | `iei` = window length in samples (end - start), not an inter-event interval |

```mermaid
flowchart LR
    A["band trace"] --> B{"rises through open level?"}
    B -- yes --> C["window start"]
    C --> D{"falls through close level?"}
    D -- yes --> E["window end"]
    E --> B
```

- Band names are `clu{position}_{max f}Hz`; position = order in the band-id list, not the band id

In [5]:
reduced = model._reduce(cluster, coeff, freqs)
band_names = list(reduced["event_onsets"].keys())

rows = []
for i, name in enumerate(band_names):
    f_lo, f_hi = reduced["reduced_freqs"][i]
    sd = float(np.std(reduced["reduced_coeff"][i]))
    window = 41 if f_hi <= 15 else 21 if f_hi <= 50 else 11 if f_hi <= 75 else "none"
    up, down = (2.0 * sd, 1.0 * sd) if f_hi < 30 else (2.5 * sd, 0.0)
    lengths = reduced["event_onsets"][name]["iei"]
    rows.append([name, f"{f_lo:.1f} - {f_hi:.1f}", window, f"{up:.3f} / {down:.3f}", len(lengths),
                 f"{1000 * np.median(lengths) / fs_hz:.1f}",
                 f"{1000 * lengths.min() / fs_hz:.1f} - {1000 * lengths.max() / fs_hz:.1f}"])
md_table(["band", "Hz", "smoothing window (samples)", "open / close level", "windows in 60 s",
          "median window (ms)", "window range (ms)"], rows)

order = np.argsort(reduced["reduced_freqs"][:, 1])
show = [order[0], order[len(order) // 2], order[-1]]
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                    subplot_titles=[f"{band_names[i]}: {reduced['reduced_freqs'][i][0]:.1f} - {reduced['reduced_freqs'][i][1]:.1f} Hz"
                                    for i in show])
for r, i in enumerate(show, start=1):
    band_trace = reduced["reduced_coeff"][i]
    fig.add_trace(go.Scatter(x=t_ex, y=band_trace[EX], mode="lines", name=band_names[i],
                             line={"width": 1, "color": "#1b998b"}), row=r, col=1)
    f_hi = reduced["reduced_freqs"][i][1]
    open_level = (2.0 if f_hi < 30 else 2.5) * float(np.std(band_trace))
    fig.add_hline(y=open_level, line={"dash": "dash", "width": 1, "color": "#333333"}, row=r, col=1)
    for st, ed in reduced["event_onsets"][band_names[i]]["crossings"]:
        if ed >= EX.start and st <= EX.stop:
            fig.add_vrect(x0=t[max(int(st), EX.start)], x1=t[min(int(ed), EX.stop - 1)],
                          fillcolor="red", opacity=0.2, line_width=0, row=r, col=1)
fig.update_xaxes(title_text="time (s)", row=3, col=1)
fig.update_layout(height=720, template="plotly_white",
                  title="Stage 4: band traces (excerpt); dashed = open level; red = event windows")
fig.show()

| band | Hz | smoothing window (samples) | open / close level | windows in 60 s | median window (ms) | window range (ms) |
|---|---|---|---|---|---|---|
| clu0_10.8Hz | 6.6 - 10.8 | 41 | 1.539 / 0.770 | 36 | 46.5 | 21.4 - 56.7 |
| clu1_6.2Hz | 3.1 - 6.2 | 41 | 1.601 / 0.801 | 32 | 75.8 | 41.9 - 94.0 |
| clu2_2.9Hz | 1.1 - 2.9 | 41 | 1.610 / 0.805 | 16 | 170.2 | 110.7 - 227.9 |
| clu3_20.1Hz | 11.5 - 20.1 | 21 | 1.261 / 0.631 | 81 | 18.6 | 10.2 - 36.3 |
| clu4_352.0Hz | 266.3 - 352.0 | none | 1.071 / 0.000 | 376 | 0.9 | 0.9 - 2.8 |
| clu5_1075.0Hz | 377.5 - 1075.0 | none | 0.460 / 0.000 | 309 | 0.9 | 0.9 - 2.8 |
| clu6_30.6Hz | 21.6 - 30.6 | 21 | 1.175 / 0.000 | 68 | 14.9 | 10.2 - 34.4 |
| clu7_57.4Hz | 32.8 - 57.4 | 11 | 1.242 / 0.000 | 107 | 8.4 | 5.6 - 12.1 |
| clu8_107.5Hz | 61.5 - 107.5 | none | 1.188 / 0.000 | 192 | 3.7 | 2.8 - 5.6 |
| clu9_248.3Hz | 115.3 - 248.3 | none | 1.057 / 0.000 | 289 | 1.9 | 0.9 - 3.7 |

## 7. Stage 5: masks and sum (`AdaptiveThreshold.run`)

| Step | Operation | soft (`Denoiser` default) | hard (`ref/` default) |
|---|---|---|---|
| 5a | starting mask value for every sample of a band | 0.7 if max f < 5 Hz; 0.5 if < 30 Hz; 0.2 if < 80 Hz; 0.1 otherwise | 0.05 everywhere (`ref/`: 0) |
| 5b | extend every window on both sides | median window length x 2.0 (max f <= 20 Hz), x 1.5 (<= 50 Hz), x 0.5 (> 50 Hz) | same |
| 5c | mask = 1 inside every extended window | | |
| 5d | Gaussian smoothing of the mask (soft edges) | sigma 10 samples if max f <= 120 Hz, else 3 | sigma 10, else 1 |
| 5e | amplitude correction | divide the band trace by sqrt(mean scale of the band) | same |
| 5f | multiply the corrected band trace by its smoothed mask | | |
| 5g | sum over bands | `rescaled_signal` (1 x N) | |

- Outside its windows a band keeps a fraction (soft) or almost nothing (hard) of its trace; inside, all of it
- The amplitude correction compensates for the wavelet transform's growth with scale, so slow and fast bands add on a comparable footing
- `thresConfig` default is `"hard"`, but `Denoiser` overrides it to `"soft"` unless told otherwise

In [6]:
masked_soft = model._adaptive_threshold(reduced)  # Denoiser default: thresConfig(thres_type="soft")
masked_hard = AdaptiveThreshold(reduced, thresConfig(thres_type="hard")).run()

rows = []
for i, name in enumerate(band_names):
    f_hi = reduced["reduced_freqs"][i][1]
    start_soft = 0.7 if f_hi < 5 else 0.5 if f_hi < 30 else 0.2 if f_hi < 80 else 0.1
    median_len = np.median(reduced["event_onsets"][name]["iei"])
    factor = 2.0 if f_hi <= 20 else 1.5 if f_hi <= 50 else 0.5
    extra = int(median_len * factor)
    sigma = 10 if f_hi <= 120 else 3
    inside = float(np.mean(masked_soft["clu_label"][i] >= 1.0))
    rows.append([name, f"{f_hi:.1f}", start_soft, f"{extra} ({1000 * extra / fs_hz:.0f} ms)", sigma,
                 f"{np.sqrt(reduced['reduced_scale'][i]):.2f}", f"{inside:.3f}"])
md_table(["band", "max Hz", "starting mask (soft)", "window extension, samples (ms)", "mask smoothing sigma (samples)",
          "amplitude divisor sqrt(scale)", "fraction of time inside a window"], rows)

masks = masked_soft["clu_label"][:, EX]
col_step = max(1, int(np.ceil(masks.shape[1] / 700)))
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.5, 0.5], vertical_spacing=0.1,
                    subplot_titles=("Stage 5: mask before smoothing (1 = inside an extended window)",
                                    "sum of masked, amplitude-corrected band traces = rescaled_signal"))
fig.add_trace(go.Heatmap(x=t_ex[::col_step], y=band_names, z=masks[:, ::col_step], colorscale="Blues",
                         showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t_ex, y=np.real(masked_soft["rescaled_signal"][0][EX]), mode="lines", name="soft",
                         line={"width": 1, "color": "#d62728"}), row=2, col=1)
fig.add_trace(go.Scatter(x=t_ex, y=np.real(masked_hard["rescaled_signal"][0][EX]), mode="lines", name="hard",
                         line={"width": 1, "color": "#4c78a8"}), row=2, col=1)
fig.update_xaxes(title_text="time (s)", row=2, col=1)
fig.update_layout(height=640, template="plotly_white", title="Stage 5 on the excerpt")
fig.show()

| band | max Hz | starting mask (soft) | window extension, samples (ms) | mask smoothing sigma (samples) | amplitude divisor sqrt(scale) | fraction of time inside a window |
|---|---|---|---|---|---|---|
| clu0_10.8Hz | 10.8 | 0.5 | 100 (93 ms) | 10 | 11.37 | 0.123 |
| clu1_6.2Hz | 6.2 | 0.5 | 163 (152 ms) | 10 | 15.93 | 0.189 |
| clu2_2.9Hz | 2.9 | 0.7 | 366 (340 ms) | 10 | 25.33 | 0.217 |
| clu3_20.1Hz | 20.1 | 0.5 | 30 (28 ms) | 10 | 8.47 | 0.102 |
| clu4_352.0Hz | 352.0 | 0.1 | 0 (0 ms) | 3 | 1.88 | 0.008 |
| clu5_1075.0Hz | 1075.0 | 0.1 | 0 (0 ms) | 3 | 1.33 | 0.005 |
| clu6_30.6Hz | 30.6 | 0.2 | 24 (22 ms) | 10 | 6.49 | 0.068 |
| clu7_57.4Hz | 57.4 | 0.2 | 4 (4 ms) | 10 | 5.02 | 0.028 |
| clu8_107.5Hz | 107.5 | 0.1 | 2 (2 ms) | 10 | 3.67 | 0.025 |
| clu9_248.3Hz | 248.3 | 0.1 | 1 (1 ms) | 3 | 2.56 | 0.019 |

## 8. Stage 6: baseline and final trace (`Denoiser.run`)

| Step | Operation | Parameter |
|---|---|---|
| 6a | slow baseline of z | FIR low-pass, cutoff 1 Hz, Hamming window 2000 ms (2149 taps at 1075 Hz), run forward and backward so nothing shifts in time |
| 6b | package output | `denoised = rescaled_signal + baseline` |
| 6c | package events | `event_indices` = sorted start samples of all windows of all bands |
| 6d | upstream stored output | `denoised_trace_scans.pkl = real(rescaled_signal)`; baseline stored separately as `lp_FIR1Hz` and **not** added |

Important

- The trace that `curation.ipynb` loads has no slow baseline; it is the masked wavelet sum only
- The stored sum is not reproduced exactly by any code version in this repo. Best match: soft masks, correlation about 0.93-0.97. The processing code was edited after these files were made (comments dated 8/28/25 and 10/1/25 in `denoiser.py`; `docs/ref_parity_audit.md`)
- The 1 Hz baseline and the wavelet coefficients are reproduced exactly, so the differences sit in stages 4-5 only

In [7]:
baseline_1hz = model._fir_lowpass(z)
recon_soft = np.real(masked_soft["rescaled_signal"][0])
recon_hard = np.real(masked_hard["rescaled_signal"][0])
package_denoised = recon_soft + baseline_1hz

rows = [
    ["fresh soft reconstruction vs stored denoised_trace_scans", f"{corr(recon_soft, stored_denoised):.3f}",
     f"{np.abs(recon_soft - stored_denoised).max():.3f}"],
    ["fresh hard reconstruction vs stored denoised_trace_scans", f"{corr(recon_hard, stored_denoised):.3f}",
     f"{np.abs(recon_hard - stored_denoised).max():.3f}"],
    ["package output (soft + 1 Hz baseline) vs stored denoised_trace_scans", f"{corr(package_denoised, stored_denoised):.3f}",
     f"{np.abs(package_denoised - stored_denoised).max():.3f}"],
    ["SD of z / fresh soft reconstruction / stored trace",
     f"{z.std():.3f} / {recon_soft.std():.3f} / {stored_denoised.std():.3f}", "-"],
]
if components is not None:
    stored_recon = np.real(np.asarray(components["rescaled_signal"])).ravel()
    stored_lp = np.asarray(components["lp_FIR1Hz"], dtype=float).ravel()
    n_differ = int(np.sum(np.abs(stored_denoised - stored_recon) > 0.01))
    rows += [
        ["fresh 1 Hz baseline vs stored lp_FIR1Hz", f"{corr(baseline_1hz, stored_lp):.5f}",
         f"{np.abs(baseline_1hz - stored_lp).max():.1e}"],
        ["stored denoised_trace_scans vs stored real(rescaled_signal)", f"{corr(stored_denoised, stored_recon):.5f}",
         f"{np.abs(stored_denoised - stored_recon).max():.3f} ({n_differ} samples differ by more than 0.01)"],
        ["stored denoised_trace_scans vs stored real(rescaled_signal) + lp_FIR1Hz",
         f"{corr(stored_denoised, stored_recon + stored_lp):.3f}", f"{np.abs(stored_denoised - stored_recon - stored_lp).max():.3f}"],
        ["fresh soft reconstruction vs stored rescaled_signal", f"{corr(recon_soft, stored_recon):.3f}",
         f"{np.abs(recon_soft - stored_recon).max():.3f}"],
        ["fresh hard reconstruction vs stored rescaled_signal", f"{corr(recon_hard, stored_recon):.3f}",
         f"{np.abs(recon_hard - stored_recon).max():.3f}"],
        ["stored rescaled_signal: max |imaginary part|", f"{np.abs(np.imag(np.asarray(components['rescaled_signal']))).max():.3f}",
         "upstream kept complex values for bands above 75 Hz; the port takes the real part first"],
    ]
md_table(["Comparison", "correlation", "max abs difference"], rows)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_ex, y=z[EX], mode="lines", name="z (denoiser input)", line={"width": 1, "color": "#bbbbbb"}))
fig.add_trace(go.Scatter(x=t_ex, y=stored_denoised[EX], mode="lines", name="stored denoised_trace_scans", line={"width": 1.5, "color": "#111111"}))
fig.add_trace(go.Scatter(x=t_ex, y=recon_soft[EX], mode="lines", name="fresh soft reconstruction", line={"width": 1, "color": "#d62728"}))
fig.add_trace(go.Scatter(x=t_ex, y=baseline_1hz[EX], mode="lines", name="1 Hz baseline (not in the stored trace)", line={"width": 1.5, "dash": "dash", "color": "#1f77b4"}))
fig.update_xaxes(title_text="time (s)")
fig.update_yaxes(title_text="z units")
fig.update_layout(height=420, template="plotly_white", title="Stage 6 on the excerpt", hovermode="x unified")
fig.show()

| Comparison | correlation | max abs difference |
|---|---|---|
| fresh soft reconstruction vs stored denoised_trace_scans | 0.929 | 0.682 |
| fresh hard reconstruction vs stored denoised_trace_scans | 0.854 | 1.062 |
| package output (soft + 1 Hz baseline) vs stored denoised_trace_scans | 0.865 | 0.718 |
| SD of z / fresh soft reconstruction / stored trace | 1.000 / 0.174 / 0.157 | - |
| fresh 1 Hz baseline vs stored lp_FIR1Hz | 1.00000 | 4.3e-05 |
| stored denoised_trace_scans vs stored real(rescaled_signal) | 0.99997 | 0.276 (1 samples differ by more than 0.01) |
| stored denoised_trace_scans vs stored real(rescaled_signal) + lp_FIR1Hz | 0.914 | 0.387 |
| fresh soft reconstruction vs stored rescaled_signal | 0.929 | 0.690 |
| fresh hard reconstruction vs stored rescaled_signal | 0.854 | 1.053 |
| stored rescaled_signal: max |imaginary part| | 0.928 | upstream kept complex values for bands above 75 Hz; the port takes the real part first |

## 9. Stage 6c: the event list returned by `Denoiser.run`

| Quantity | Definition |
|---|---|
| `event_indices` | start sample of every window of every band, concatenated and sorted |
| `event_ranges_` | per band: (start, end) sample pairs |
| duplicates | one real event opens windows in several bands, so it appears several times |
| use | none in the curation workflow; `pipeline.ipynb` and the raw-`.mat` path replace it with a single PCA score (section 12) |

In [8]:
starts, ranges = model._collect_event_indices(reduced["event_onsets"])
gaps = np.diff(starts)
md_table(["Quantity", "Value"], [
    ["window starts, all bands", len(starts)],
    ["distinct start samples", len(np.unique(starts))],
    ["starts within 6 ms of the previous start", int(np.sum(gaps <= 0.006 * fs_hz))],
    ["upstream detected peaks in this trace (section 10)", len(load_pickle(PF / "detected_events_peaks.pkl")[SCAN][DOMAIN])],
])
md_table(["band", "windows"], [[name, len(windows)] for name, windows in ranges.items()])

| Quantity | Value |
|---|---|
| window starts, all bands | 1506 |
| distinct start samples | 1427 |
| starts within 6 ms of the previous start | 331 |
| upstream detected peaks in this trace (section 10) | 37 |

| band | windows |
|---|---|
| clu0_10.8Hz | 36 |
| clu1_6.2Hz | 32 |
| clu2_2.9Hz | 16 |
| clu3_20.1Hz | 81 |
| clu4_352.0Hz | 376 |
| clu5_1075.0Hz | 309 |
| clu6_30.6Hz | 68 |
| clu7_57.4Hz | 107 |
| clu8_107.5Hz | 192 |
| clu9_248.3Hz | 289 |

## 10. Stage 8: upstream peak detector (**inferred**; code not in repo)

| Parameter (`param_spike_detect.pkl`) | Value | Reading |
|---|---|---|
| `bp` | [3, 400] | band-pass 3-400 Hz applied to the stored trace |
| `thres_bp_sd` | 3.5 | peak must exceed 3.5 SD of the band-passed trace |
| `thres_amp_sd` | 3.5 | peak must exceed 3.5 SD on the stored trace |
| `duration_thres` | 5 | minimum duration in samples (4.7 ms) |

- Output: `detected_events_peaks.pkl[scan][domain]` = peak sample indices; `..._curated.pkl` = subset after manual review
- Checks below test the reading against the saved peaks (4th-order Butterworth band-pass used as a stand-in; the real filter is unknown)
- `curation.ipynb` never reads these files: the loader is called with `load_events=False`

In [9]:
params = load_pickle(PF / "param_spike_detect.pkl")
peaks = np.asarray(load_pickle(PF / "detected_events_peaks.pkl")[SCAN][DOMAIN], dtype=int)
peaks_curated = np.asarray(load_pickle(PF / "detected_events_peaks_curated.pkl")[SCAN][DOMAIN], dtype=int)
sos = butter(4, params["bp"], btype="band", fs=fs_hz, output="sos")
bandpassed = sosfiltfilt(sos, stored_denoised)

md_table(["Check on scan 46 soma", "Value"], [
    ["peaks: detected / curated", f"{len(peaks)} / {len(peaks_curated)}"],
    ["curated is a subset of detected", bool(np.isin(peaks_curated, peaks).all())],
    ["fraction of peaks that are local maxima of the 3-400 Hz band-passed stored trace", f"{np.isin(peaks, find_peaks(bandpassed)[0]).mean():.2f}"],
    ["fraction of peaks that are local maxima of the stored trace itself", f"{np.isin(peaks, find_peaks(stored_denoised)[0]).mean():.2f}"],
    ["smallest band-passed value at a peak, in SD of the band-passed trace", f"{(bandpassed[peaks] / bandpassed.std()).min():.2f}"],
    ["smallest stored-trace value at a peak, in SD of the stored trace", f"{(stored_denoised[peaks] / stored_denoised.std()).min():.2f}"],
    ["smallest spacing between peaks (ms)", f"{1000 * np.diff(np.sort(peaks)).min() / fs_hz:.1f}"],
])

| Check on scan 46 soma | Value |
|---|---|
| peaks: detected / curated | 37 / 37 |
| curated is a subset of detected | True |
| fraction of peaks that are local maxima of the 3-400 Hz band-passed stored trace | 1.00 |
| fraction of peaks that are local maxima of the stored trace itself | 0.78 |
| smallest band-passed value at a peak, in SD of the band-passed trace | 3.68 |
| smallest stored-trace value at a peak, in SD of the stored trace | 3.56 |
| smallest spacing between peaks (ms) | 2.8 |

## 11. What `curation.ipynb` does at Load (`vnoiser/curation.py`)

```mermaid
flowchart TD
    A["Select animal / experiment / scan / domain"] --> B["SpatialJediDataset.load: trace = denoised_trace_scans.pkl[scan][domain], load_events=False"]
    B --> C{"metadata pre_denoised?"}
    C -- "yes (spatial JEDI pickles)" --> D["Denoiser NOT run; denoised = stored trace"]
    C -- "no (raw .mat)" --> E["Denoiser with the raw-.mat settings + PCA event score (cached)"]
    D --> F{"mode"}
    E --> F
    F -- "manual / fast" --> G["analysis trace = denoised"]
    F -- "slow" --> H["analysis trace = 40 Hz low-pass of denoised"]
    G --> I["threshold slider: default = median + 3 robust SD"]
    H --> I
    I --> J["candidates = local maxima above threshold, min spacing 6 ms / 100 ms, plus retained manual events"]
    J --> K["per candidate: align to local peak, short snippet, amplitude, long snippet"]
    K --> L["fast / slow: seed template = mean of top 25 percent amplitude; cosine per candidate; auto call"]
    K --> M["PCA panel: 400 ms windows -> PC1, PC2"]
    L --> N["Yes / No clicks -> template update, second-pass preview, JSON write"]
    M --> N
```

| Step | manual | fast | slow |
|---|---|---|---|
| analysis trace | stored trace | stored trace | 4th-order Butterworth low-pass 40 Hz, forward and backward (`lowpass_trace`) |
| default threshold | median + 3 robust SD; robust SD = 1.4826 x MAD | same | same, on the low-passed trace |
| slider range | median ... 99th percentile of local maxima + 5 robust SD (capped at the largest maximum) | same | same |
| candidate | local maximum above threshold, min spacing 6 ms (`find_peaks`) | same | min spacing 100 ms |
| retained events | events labelled Yes/No stay listed when below threshold | same | same |
| peak alignment search | +-3 ms around the candidate | same | +-25 ms |
| polarity | sign of the median trace value at the candidates | same | same |
| short snippet | -4 to +10 ms around the aligned peak, minus the median of the 4 ms before | same | -500 to +500 ms, minus the median of the 500 ms before |
| amplitude | short snippet value at the aligned peak | same | same |
| long snippet | +-500 ms of the stored trace, minus the median of the 500 ms before | same | +-500 ms of the low-passed trace |
| PCA panel | long snippets cropped to +-200 ms, mean removed, 2 components | same | same |
| seed template | none | mean short snippet of the top 25 % amplitude candidates (ceil(0.25 n), at least 1) | same |
| automatic call | none | amplitude >= auto-pass slider -> pass; else cosine > 0.80 -> pass; else reject when waveform rejection is on | same |
| evolving template | mean of Yes events | seeds not labelled No, plus Yes events | same |
| second-pass preview | No events replaced by a straight line over -4 to +12 ms | same | -500 to +500 ms |
| saved file | `PF/.curation/manual_template_curation.json` | `fast_template_curation.json` | `slow_template_curation.json` |

Fields saved per labelled event (`_save_labels`)

| Field | Content |
|---|---|
| `label`, `manual_label`, `curation_state`, `mode` | Yes/No and the mode that wrote it |
| `recording`, `filename` | `animal/experiment/scan=../domain=..` id and trace file name |
| `source_event_index`, `source_aligned_index`, `source_event_time_s` | candidate sample, aligned peak sample, time in the full trace |
| `window_*` | same, relative to the loaded window (equal to source values when `duration_s=None`) |
| `amplitude`, `candidate_source`, `candidate_threshold` | snippet amplitude; `threshold` or `retained_manual`; slider value at save time |
| `template_cosine`, `initial_template_cosine`, `initial_auto_call` | similarity to the evolving and seed templates; automatic call at save time |
| `auto_template_threshold`, `auto_pass_amplitude`, `waveform_rejection`, `was_seed_template_source` | settings that produced the automatic call |
| `updated_utc` | timestamp |

File-level fields: `version`, `mode`, `data_path`, `candidate_detection` (per-recording threshold, auto-pass amplitude, waveform-rejection flag, source trace, slow cutoff), `updated_utc`, `events`.

- Fast-mode `amplitude` is the rise over the 4 ms before the aligned peak (4 samples at 1075 Hz), not the peak height. Seed selection (top 25 %) and the auto-pass slider rank by this value; compare the two amplitude rows in the output below

In [10]:
FAST_MIN_DISTANCE_MS = 6.0     # EventCurationDashboard defaults
SLOW_MIN_DISTANCE_MS = 100.0
SLOW_CUTOFF_HZ = 40.0
HIGH_AMPLITUDE_QUANTILE = 0.75

trace = stored_denoised                                             # metadata["pre_denoised"] is True: Denoiser not run
analysis_fast = trace.copy()                                        # manual and fast modes
analysis_slow = lowpass_trace(trace, fs_hz, cutoff_hz=SLOW_CUTOFF_HZ)  # slow mode

detected = {}
rows = []
for mode, analysis, min_distance in [("manual / fast", analysis_fast, FAST_MIN_DISTANCE_MS),
                                     ("slow", analysis_slow, SLOW_MIN_DISTANCE_MS)]:
    lower, upper, selected, step = threshold_slider_scale(analysis, fs_hz, min_distance)
    center = float(np.median(analysis))
    sigma = float(1.4826 * np.median(np.abs(analysis - center)))
    idx = threshold_event_calls(analysis, fs_hz, selected, min_distance)
    detected[mode] = (idx, selected)
    rows.append([mode, f"{center:.3f}", f"{sigma:.3f}", f"{selected:.3f}", f"{lower:.3f} - {upper:.3f}", min_distance, len(idx)])
md_table(["mode", "median of trace", "robust SD", "default threshold", "slider range", "min spacing (ms)", "candidates"], rows)


def build_candidates(analysis, long_source, indices, fs_hz, short_pre_s, short_post_s, align_s, long_half_s=0.5):
    # Mirror of EventCurationDashboard._build_candidates for one mode.
    short_pre = max(1, int(round(short_pre_s * fs_hz)))
    short_post = max(1, int(round(short_post_s * fs_hz)))
    align = max(1, int(round(align_s * fs_hz)))
    long_pre = long_post = max(1, int(round(long_half_s * fs_hz)))
    polarity = 1 if np.nanmedian(analysis[indices]) >= 0 else -1
    kept, aligned, amplitudes, short_snips, long_snips = [], [], [], [], []
    for idx in np.asarray(indices, dtype=int):
        lo, hi = max(0, idx - align), min(len(analysis), idx + align + 1)
        peak = lo + int(np.argmax(polarity * analysis[lo:hi]))
        if peak - short_pre < 0 or peak + short_post + 1 > len(analysis):
            continue
        short = polarity * analysis[peak - short_pre:peak + short_post + 1].copy()
        short -= np.median(short[:short_pre])
        long = np.full(long_pre + long_post + 1, np.nan)
        src_lo, src_hi = max(0, peak - long_pre), min(len(long_source), peak + long_post + 1)
        dst_lo = src_lo - (peak - long_pre)
        baseline = np.median(long_source[src_lo:max(src_lo + 1, peak)])
        long[dst_lo:dst_lo + (src_hi - src_lo)] = polarity * (long_source[src_lo:src_hi] - baseline)
        kept.append(idx); aligned.append(peak); amplitudes.append(short[short_pre])
        short_snips.append(short); long_snips.append(long)
    short_snips, long_snips = np.asarray(short_snips), np.asarray(long_snips)
    return {
        "indices": np.asarray(kept, dtype=int), "aligned": np.asarray(aligned, dtype=int),
        "amplitudes": np.asarray(amplitudes, dtype=float), "short": short_snips, "long": long_snips,
        "short_time_ms": (np.arange(short_snips.shape[1]) - short_pre) / fs_hz * 1000.0,
        "long_time_ms": (np.arange(long_snips.shape[1]) - long_pre) / fs_hz * 1000.0,
        "polarity": polarity,
    }


fast_idx, fast_threshold = detected["manual / fast"]
fast = build_candidates(analysis_fast, trace, fast_idx, fs_hz, 0.004, 0.010, 0.003)
n_seed = min(len(fast["indices"]), max(1, int(np.ceil(len(fast["indices"]) * (1.0 - HIGH_AMPLITUDE_QUANTILE)))))
seed = np.sort(np.argsort(fast["amplitudes"])[::-1][:n_seed])
template = fast["short"][seed].mean(axis=0)
cosine = cosine_similarity_rows(fast["short"], template)
auto_pass = cosine > AUTO_TEMPLATE_THRESHOLD
pca_scores, pca_var = candidate_pca_embedding(fast["long"], fast["long_time_ms"], window_ms=400.0)

slow_idx, slow_threshold = detected["slow"]
slow = build_candidates(analysis_slow, analysis_slow, slow_idx, fs_hz, 0.500, 0.500, 0.025)

matched = np.array([np.min(np.abs(fast["aligned"] - p)) <= 0.003 * fs_hz for p in peaks]) if len(fast["aligned"]) else np.zeros(len(peaks), bool)
md_table(["Fast mode at Load", "Value"], [
    ["candidates (local maxima above the default threshold)", len(fast["indices"])],
    ["seed template: top 25 % by amplitude", n_seed],
    ["automatic call at cosine 0.80: pass / reject", f"{int(auto_pass.sum())} / {int((~auto_pass).sum())}"],
    ["cosine to seed template: min / median / max", f"{cosine.min():.2f} / {np.median(cosine):.2f} / {cosine.max():.2f}"],
    ["amplitude: min / median / max", f"{fast['amplitudes'].min():.2f} / {np.median(fast['amplitudes']):.2f} / {fast['amplitudes'].max():.2f}"],
    ["trace value at the aligned peak: min / median / max", f"{trace[fast['aligned']].min():.2f} / {np.median(trace[fast['aligned']]):.2f} / {trace[fast['aligned']].max():.2f}"],
    ["PCA panel: variance explained by PC1 / PC2", f"{pca_var[0]:.2f} / {pca_var[1]:.2f}"],
    ["upstream detected peaks within 3 ms of a fast candidate", f"{int(matched.sum())} / {len(peaks)}"],
    ["slow mode candidates (40 Hz low-pass, 100 ms spacing)", len(slow["indices"])],
])

colors = np.where(auto_pass, "#2ca02c", "#d62728")
in_ex = (fast["aligned"] >= EX.start) & (fast["aligned"] < EX.stop)
fig = make_subplots(rows=1, cols=3, column_widths=[0.5, 0.25, 0.25], horizontal_spacing=0.08,
                    subplot_titles=("Panel A: stored trace, threshold, candidates (excerpt)",
                                    "Panel B: seed snippets and template",
                                    "Panel E: PCA of +-200 ms windows"))
fig.add_trace(go.Scatter(x=t_ex, y=trace[EX], mode="lines", name="stored trace", line={"width": 1, "color": "#111111"}), row=1, col=1)
fig.add_trace(go.Scatter(x=t_ex, y=analysis_slow[EX], mode="lines", name="40 Hz low-pass (slow mode)", line={"width": 1, "color": "#1f77b4"}), row=1, col=1)
fig.add_hline(y=fast_threshold, line={"dash": "dash", "width": 1, "color": "#333333"}, row=1, col=1)
fig.add_trace(go.Scatter(x=t[fast["aligned"][in_ex]], y=trace[fast["aligned"][in_ex]], mode="markers", name="candidate (green = auto pass, red = auto reject)",
                         marker={"color": colors[in_ex], "size": 8}), row=1, col=1)
peaks_ex = peaks[(peaks >= EX.start) & (peaks < EX.stop)]
fig.add_trace(go.Scatter(x=t[peaks_ex], y=np.full(len(peaks_ex), trace[EX].max() * 1.1), mode="markers", name="upstream detected peak",
                         marker={"symbol": "line-ns-open", "size": 10, "color": "#ff7f0e"}), row=1, col=1)
for i in seed:
    fig.add_trace(go.Scatter(x=fast["short_time_ms"], y=fast["short"][i], mode="lines", showlegend=False, hoverinfo="skip",
                             line={"width": 1, "color": "#4c78a8"}, opacity=0.25), row=1, col=2)
fig.add_trace(go.Scatter(x=fast["short_time_ms"], y=template, mode="lines", name="seed template", line={"width": 3, "color": "#111111"}), row=1, col=2)
fig.add_trace(go.Scatter(x=pca_scores[:, 0], y=pca_scores[:, 1], mode="markers", name="candidates in PCA space",
                         marker={"color": colors, "size": 6}), row=1, col=3)
fig.update_xaxes(title_text="time (s)", row=1, col=1)
fig.update_xaxes(title_text="ms from aligned peak", row=1, col=2)
fig.update_xaxes(title_text="PC1", row=1, col=3)
fig.update_yaxes(title_text="PC2", row=1, col=3)
fig.update_layout(height=430, template="plotly_white", title="Curation Load state, fast mode, before any click",
                  legend={"orientation": "h", "y": -0.25})
fig.show()

| mode | median of trace | robust SD | default threshold | slider range | min spacing (ms) | candidates |
|---|---|---|---|---|---|---|
| manual / fast | -0.014 | 0.091 | 0.260 | -0.014 - 1.040 | 6.0 | 119 |
| slow | -0.014 | 0.089 | 0.253 | -0.014 - 1.483 | 100.0 | 77 |

| Fast mode at Load | Value |
|---|---|
| candidates (local maxima above the default threshold) | 119 |
| seed template: top 25 % by amplitude | 30 |
| automatic call at cosine 0.80: pass / reject | 106 / 13 |
| cosine to seed template: min / median / max | -0.90 / 0.95 / 0.98 |
| amplitude: min / median / max | -0.03 / 0.03 / 0.62 |
| trace value at the aligned peak: min / median / max | 0.26 / 0.37 / 1.46 |
| PCA panel: variance explained by PC1 / PC2 | 0.40 / 0.08 |
| upstream detected peaks within 3 ms of a fast candidate | 37 / 37 |
| slow mode candidates (40 Hz low-pass, 100 ms spacing) | 76 |

## 12. The raw-`.mat` path (only when `DATA_PATH` points at `.mat` files)

| Step | Operation | Value |
|---|---|---|
| input | `CAttached.fluo_mean`, `fluo_time`; optional `events_AP` | fs = 1 / median frame interval |
| scaling | (trace - median) / SD | not the stage-1 z-score; no sign flip |
| Denoiser settings | 64 scales log-spaced 2 ... 1800; baseline 20 Hz, 80 ms FIR; PCA 16 / 8 used; 5 clusters / 8 bands; open levels 1.5 / 2.0 SD; soft masks | `EventCurationDashboard._run_pipeline` |
| output | `denoised = rescaled_signal + 20 Hz baseline` | baseline **is** added on this path |
| PCA event score | robust z of each masked band trace -> PC1 -> sign matched to the input -> 1.5 ms Gaussian smoothing -> robust z -> peaks above 3, prominence 1, spacing 6 ms | `consolidated_pca_event_calls` |
| use of the PCA score | cached in `.curation/cache/*.npz`; candidates still come from the threshold on `denoised` | |
| then | same candidate, template, and label steps as section 11 | |

## 13. Parameter summary across the three configurations in the repo

| Parameter | Upstream run = `Denoiser` defaults | `curation.ipynb` on raw `.mat` | `pipeline.ipynb` |
|---|---|---|---|
| input | stage-1 z (sign flipped) | (raw - median) / SD | (raw - median) / SD |
| scales | 100, log 1 ... 1000 | 64, log 2 ... 1800 | 64, log 2 ... 1800 |
| row frequencies at fs = 1075 Hz | 1.08 ... 1075 Hz | 0.6 ... 537 Hz | 0.6 ... 537 Hz |
| PCA components / used / clusters / bands | 30 / 10 / 5 / 10 | 16 / 8 / 5 / 8 | 16 / 8 / 5 / 8 |
| open level, slow bands / fast bands | 2.0 / 2.5 SD | 1.5 / 2.0 SD | 1.5 / 2.0 SD |
| masks | soft (`Denoiser` default; upstream files closest to soft) | soft | soft |
| baseline | 1 Hz, 2000 ms window | 20 Hz, 80 ms window | 20 Hz, 80 ms window |
| stored / used final trace | real(sum), baseline not added | sum + baseline | sum + baseline |
| event list used next | separate 3-400 Hz peak detector (not read by curation) | threshold candidates on `denoised` | PCA score peaks + template cosine |

## 14. Three different event lists

| List | Where it is made | Rule | Reaches the dashboard |
|---|---|---|---|
| `Denoiser` `event_indices` | `Denoiser.run` | starts of per-band windows; duplicates across bands | no |
| `detected_events_peaks.pkl` | upstream, code not in repo | 3-400 Hz band-pass, 3.5 SD, local maxima (**inferred**) | no (`load_events=False`) |
| curation candidates | `curation.py` at Load and on every slider move | local maxima above the slider threshold on the stored trace (or its 40 Hz low-pass) | yes |

## 15. Glossary

| Term | Meaning here |
|---|---|
| wavelet transform | a set of band-pass filters whose width scales with frequency; output = one filtered trace per frequency row, complex-valued |
| magnitude / real part | magnitude = strength of the oscillation at that time; real part = the filtered signal with its sign |
| scale | stretch factor of the wavelet; row frequency = fs / scale for this wavelet |
| PCA | finds the few directions that carry most of the variation; used to compress each row's time course before grouping, and to place candidate snippets in a 2-D plot |
| Ward linkage | merges groups so that the spread inside groups grows as little as possible |
| Savitzky-Golay, order 1 | smoothing by a running straight-line fit |
| SD | standard deviation |
| robust SD | 1.4826 x median absolute deviation; an SD estimate that events do not inflate |
| FIR low-pass, forward and backward | a finite-length smoothing filter run twice in opposite directions so the output does not shift in time |
| soft / hard mask | soft: a band keeps 10-70 % of its trace between events; hard: about 5 % |
| cosine similarity | shape match between two snippets after removing their means; 1 = same shape, 0 = unrelated, negative = inverted |
| local maximum | a sample larger than both neighbours (`scipy.signal.find_peaks`) |

## 16. Findings from this review

- `denoised_trace_scans.pkl` holds the wavelet sum without the 1 Hz baseline; `Denoiser.run` adds the baseline. Curated traces and freshly denoised traces therefore differ by a slow component
- No code version in the repo reproduces the stored sum exactly (correlation 0.93-0.97 with the closest variant); coefficients and baseline do reproduce exactly
- `SpatialJediDataset` builds domain names from `scanIDs_ROIs.pkl` (`soma1`, `bg`), but `denoised_trace_scans.pkl` uses `soma` and has no `bg`. Loading `soma1` or `bg` raises `KeyError`; apical domains load. Verified on this experiment
- `cwts.h5` frequency labels differ from `pywt` labels by up to 0.27 Hz (rounded sampling period upstream); coefficients are identical
- `iei` inside `event_onsets` is the window length, not an inter-event interval
- The PCA event score on the raw-`.mat` path is computed and cached but never used for candidates
- Fast-mode `amplitude` (rise over the preceding 4 ms) is much smaller than the peak value for these 1075 Hz traces; the seed template and auto-pass rule rank candidates by that rise, not by peak height